# Reconocimiento de Dígitos con Redes Neuronales Multicapa

**Clasificación multiclase con MLP, optimización y análisis de predicciones**

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luisf2020/Proyectos-de-Inteligencia-Artificial/blob/main/notebooks/07_clasificacion_digitos_mlp.ipynb)

> Implementación desarrollada específicamente para este repositorio como proyecto técnico de portafolio.

## Objetivo

Entrenar una red neuronal multicapa para reconocer dígitos escritos a mano y comparar conceptualmente su capacidad con la de un clasificador lineal. El conjunto `load_digits` mantiene el experimento ligero y completamente reproducible en Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

mlp = Pipeline([
    ("scaler", StandardScaler()),
    ("network", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        max_iter=600,
        early_stopping=True,
        validation_fraction=0.12,
        random_state=42
    ))
])

mlp.fit(X_train, y_train)
pred = mlp.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, pred):.3f}")
print(classification_report(y_test, pred, digits=3))

In [ ]:
network = mlp.named_steps["network"]
plt.figure(figsize=(8, 4))
plt.plot(network.loss_curve_)
plt.xlabel("Iteración")
plt.ylabel("Loss")
plt.title("Curva de optimización del MLP")
plt.show()

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred,
    cmap="Blues",
    colorbar=False
)
plt.title("Matriz de confusión — MLP")
plt.show()

In [ ]:
wrong = np.flatnonzero(pred != y_test)
n_show = min(12, len(wrong))

fig, axes = plt.subplots(3, 4, figsize=(9, 7))
for ax in axes.ravel():
    ax.axis("off")

for ax, idx in zip(axes.ravel(), wrong[:n_show]):
    ax.imshow(X_test[idx].reshape(8, 8), cmap="gray")
    ax.set_title(f"Real {y_test[idx]} | Pred {pred[idx]}")
    ax.axis("off")

plt.suptitle("Ejemplos de errores de clasificación")
plt.tight_layout()
plt.show()

## Conclusiones

El MLP aprende transformaciones internas de las características antes de decidir la clase final. En problemas visuales, esta capacidad suele superar a un modelo puramente lineal, aunque sigue sin explotar de manera explícita la estructura local de la imagen como lo haría una red convolucional.